<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Skill Compliance
</b></font> </br></p>

---

In [1]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M31-Agent-Skill-Compliance"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

# Run Compression deaktivieren — verhindert "pending"-Runs in Jupyter/Colab
# Ursache: LangSmith >= 0.3.42 komprimiert Traces im Hintergrund;
# der Thread bricht vorzeitig ab, bevor end_time-Events für Parent-Runs gesendet werden
os.environ["LANGSMITH_DISABLE_RUN_COMPRESSION"] = "true"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"]    = "false"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11

IP-Adresse: 35.196.78.15
Hostname: 15.78.196.35.bc.googleusercontent.com
Stadt: North Charleston
Region: South Carolina
Land: US
Koordinaten: 32.8546,-79.9748
Provider: AS396982 Google LLC
Postleitzahl: 29415
Zeitzone: America/New_Yo

# 1 | Übersicht
---

Dieses Modul zeigt, warum **Skills** ein wichtiges Mittel zur Steuerung von Agenten sind.

Ein Skill ist kein normaler Prompt, sondern ein **wiederverwendbares Arbeitsrezept**:
- wann ein bestimmtes Vorgehen aktiviert wird
- welche Prüfschritte in welcher Reihenfolge nötig sind
- welche Regeln, Referenzen und Hilfsskripte der Agent heranzieht

Im Mittelpunkt steht hier ein **Compliance-Skill**. Er demonstriert drei Kernideen:

| Idee | Bedeutung im Agenten-Kontext |
|---|---|
| **Workflow-Orchestrierung** | Der Agent arbeitet feste Prüfschritte in Reihenfolge ab |
| **Guardrails** | Kritische Prüfungen werden nicht stillschweigend übersprungen |
| **Progressive Disclosure** | Nur die Kernlogik liegt in `SKILL.md`, Details werden bei Bedarf geladen |


# 2 | Prompt vs. Skill
---

Ein **Prompt** gibt einem Agenten eine Anweisung für eine einzelne Aufgabe. Ein **Skill** ist ein wiederverwendbares Arbeitsrezept mit fester Struktur, Regeln und Hilfsmitteln.

| | Prompt | Skill |
|---|---|---|
| **Zweck** | Einzelne Antwort oder Aufgabe | Wiederverwendbarer Workflow |
| **Struktur** | Freitext | Definierter Ablauf mit Pflichtschritten |
| **Guardrails** | Keine oder ad-hoc | Explizit — was darf nicht übersprungen werden |
| **Referenzen** | Alles im Prompt | Ausgelagert in `references/` — nur bei Bedarf geladen |
| **Hilfsskripte** | Keine | `scripts/` für deterministische Teilaufgaben |
| **Wiederverwendung** | Copy-Paste | Versioniert, referenzierbar, updatebar |
| **Typischer Einsatz** | Frage beantworten, Text generieren | Compliance, Genehmigung, Onboarding, Review |



**Faustregel:** Sobald ein Prozess Pflichtschritte, Eskalationsregeln oder eine dokumentierte Entscheidung erfordert — Skill statt Prompt.

# 3 | Skill-Design für Agenten
---

<p><font color='black' size="5">Warum gehört das in einen Agenten-Kurs?</font></p>

Skills operationalisieren Fachwissen. Der Agent bekommt dadurch nicht nur Antworten, sondern eine **strukturierte Handlungslogik**.

Gerade bei Compliance-, Security- oder Freigabeprozessen reicht ein freier Prompt oft nicht aus. Man will stattdessen:
- Pflichtprüfungen definieren
- Eskalationsregeln festlegen
- Entscheidungsdokumentation erzwingen
- wiederkehrende Logik verlässlich wiederverwenden

<p><font color='black' size="5">Empfohlene Aufteilung</font></p>

| Datei-/Ordner-Typ | Rolle |
|---|---|
| `SKILL.md` | Kernablauf, Trigger, Guardrails, Verweise |
| `references/*.md` | Detailwissen, Regeln, Checklisten, Beispiele |
| `scripts/` | deterministische Hilfslogik für fragile oder wiederkehrende Schritte |

Das ist genau der Punkt, den Ihre ursprüngliche Frage adressiert hat: **Ja, Skills dürfen detailliert sein, aber die Details sollten sauber aufgeteilt werden.**

In [2]:
import urllib.request

GITHUB_BASE = "https://raw.githubusercontent.com/ralf-42/Agenten/main/06_skill/compliance"

def fetch_skill_file(path: str) -> str:
    url = f"{GITHUB_BASE}/{path}"
    with urllib.request.urlopen(url) as r:
        return r.read().decode("utf-8")

# Überblick über verfügbare Dateien im Skill
skill_files = [
    "SKILL.md",
    "references/checklist.md",
    "references/risk_rules.md",
    "references/examples.md",
    "scripts/assess_risk.py",
]
print(f"Skill-Quelle: {GITHUB_BASE}\n")
for f in skill_files:
    print(f"  {f}")

Skill-Quelle: https://raw.githubusercontent.com/ralf-42/Agenten/main/06_skill/compliance

  SKILL.md
  references/checklist.md
  references/risk_rules.md
  references/examples.md
  scripts/assess_risk.py


# 4 | Hands-On: Compliance-Skill lesen
---

In [3]:
skill_text = fetch_skill_file("SKILL.md")
print(skill_text[:3000])

---
name: compliance-skill
description: Führt compliance-orientierte Agenten-Workflows durch — Sanktionsprüfung, Risikobewertung (low/medium/high), Freigabe-Gate und auditierbare Entscheidungsnotiz. Aktivieren wenn Nutzer sagt: "prüfe diesen Lieferanten", "Sanktionsprüfung", "Onboarding prüfen", "Zahlung freigeben", "KYC/KYB-Prüfung", "Risikobewertung", "Compliance-Check", "ist dieser Kunde unbedenklich", "Zahlungsfreigabe", "Go/No-Go-Entscheidung", "due diligence" oder "AML-Prüfung".
---

# Compliance Skill

Dieser Skill definiert einen wiederverwendbaren Compliance-Workflow für Agentenprozesse mit Freigabe- und Dokumentationsanforderungen.

## Schnellstart

Diesen Skill verwenden, wenn eine Anfrage eines oder mehrere dieser Muster enthält:

- Eine Transaktion, ein Lieferant, ein Kunde oder ein Konto muss vor der Ausführung geprüft werden
- Der Agent muss eine feste Prüfsequenz einhalten, bevor er handelt
- Der Nutzer fragt nach Sanktionsprüfungen, KYC/KYB-Prüfungen, Risikobewertung, 

Achten Sie beim Lesen auf drei Dinge:
- **Trigger-Beschreibung:** Wann soll der Skill aktiv werden?
- **Core Workflow:** Welche Schritte sind verpflichtend?
- **Guardrails:** Was darf der Agent nicht behaupten oder überspringen?


In [ ]:
for name in ['checklist.md', 'risk_rules.md', 'examples.md']:
    content = fetch_skill_file(f"references/{name}")
    mprint(f'\n### {name}\n\n{content[:1500]}')

# 5 | Hands-On: deterministische Risiko-Prüfung
---

Der Skill enthält zusätzlich ein kleines Skript. Das ist nützlich, wenn ein Teil der Logik **nicht nur beschrieben, sondern zuverlässig berechnet** werden soll.

In [5]:
import json

# assess_risk.py aus GitHub laden — in eigenem Namespace ausführen,
# damit der if __name__ == "__main__" Block nicht getriggert wird
script_code = fetch_skill_file("scripts/assess_risk.py")
_ns = {"__name__": "__exec__"}
exec(script_code, _ns)
assess = _ns["assess"]  # Funktion in den Notebook-Scope holen

case = {
    'subject_type': 'vendor',
    'subject_name': 'Example Trading LLC',
    'country': 'RU',
    'transaction_amount': 18000,
    'sanctions_hit': False,
    'adverse_media_hit': True,
    'pep_flag': False,
    'documents_complete': True,
}

risk_result = assess(case)
print(json.dumps(risk_result, indent=2))

{
  "risk_score": 70,
  "risk_level": "high",
  "decision": "block",
  "reasons": [
    "adverse_media_hit",
    "high_risk_country:RU",
    "large_transaction"
  ]
}


In [6]:
from textwrap import dedent

decision_note = dedent(f'''
### Compliance Decision

- Case: {case['subject_type']} for {case['subject_name']}
- Checks performed: sanctions screening, geography review, transaction size review, documentation check
- Risk level: {risk_result['risk_level']}
- Decision: {risk_result['decision']}
- Rationale: {', '.join(risk_result['reasons'])}
- Missing information or escalation point: none in this training example
- Audit note: simulated training assessment, no live sanctions source connected
''').strip()

mprint(decision_note)

### Compliance Decision

- Case: vendor for Example Trading LLC
- Checks performed: sanctions screening, geography review, transaction size review, documentation check
- Risk level: high
- Decision: block
- Rationale: adverse_media_hit, high_risk_country:RU, large_transaction
- Missing information or escalation point: none in this training example
- Audit note: simulated training assessment, no live sanctions source connected

# 6 | Agent mit Compliance-Skill
---

Bisher haben wir den Skill manuell gelesen und das Skript direkt aufgerufen. Jetzt übergeben wir beides an einen Agenten:

- **`SKILL.md`** wird als System-Prompt geladen — der Agent kennt damit den vollständigen Compliance-Workflow
- **`assess_risk`** wird als `@tool` bereitgestellt — der Agent kann die deterministische Prüfung selbst aufrufen
- Der Agent entscheidet eigenständig, wann er das Tool einsetzt und wie er das Ergebnis kommuniziert

| Komponente | Rolle |
|---|---|
| System-Prompt (SKILL.md) | Steuert Verhalten, Pflichtschritte, Guardrails |
| `@tool assess_risk` | Deterministische Risikobewertung |
| Agent | Orchestriert beides, erzeugt Compliance Decision |

In [7]:
from langchain_core.tools import tool

@tool
def compliance_check(
    subject_type: str,
    subject_name: str,
    country: str,
    transaction_amount: float = 0,
    sanctions_hit: bool = False,
    adverse_media_hit: bool = False,
    pep_flag: bool = False,
    documents_complete: bool = True,
) -> str:
    """Führt eine deterministische Compliance-Risikoprüfung durch und gibt
    risk_score, risk_level, decision und reasons zurück."""
    case = {
        "subject_type": subject_type,
        "subject_name": subject_name,
        "country": country,
        "transaction_amount": transaction_amount,
        "sanctions_hit": sanctions_hit,
        "adverse_media_hit": adverse_media_hit,
        "pep_flag": pep_flag,
        "documents_complete": documents_complete,
    }
    return json.dumps(assess(case), indent=2)

print("Tool registriert:", compliance_check.name)

Tool registriert: compliance_check


In [8]:
ALLOWED_REFERENCES = {"checklist.md", "risk_rules.md", "examples.md"}

@tool
def load_reference(filename: str) -> str:
    """Lädt eine Referenz-Datei des Compliance-Skills.
    Erlaubte Werte: checklist.md, risk_rules.md, examples.md"""
    if filename not in ALLOWED_REFERENCES:
        return f"Unbekannte Referenz: {filename}. Erlaubt: {sorted(ALLOWED_REFERENCES)}"
    return fetch_skill_file(f"references/{filename}")

print("Tool registriert:", load_reference.name)

Tool registriert: load_reference


In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# SKILL.md von GitHub laden — Frontmatter wird automatisch entfernt
skill_system_prompt = load_prompt(f"{GITHUB_BASE}/SKILL.md", mode="S")

llm = init_chat_model(BASELINE, temperature=0.0)
memory = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[compliance_check, load_reference],
    prompt=skill_system_prompt,
    checkpointer=memory,
)

print("Agent erstellt mit create_react_agent + MemorySaver")
print("Tools:", [t.name for t in [compliance_check, load_reference]])

In [10]:
import time

# Neue Session für jeden Testlauf (verhindert Überlappung mit vorherigen Runs)
session = {"configurable": {"thread_id": f"compliance-{int(time.time())}"}}

# ── Aufruf 1: vollständiger Fließtext — sanctions_clearance_confirmed fehlt ──
user_input_1 = (
    "Wir wollen einen neuen Lieferanten onboarden: "
    "Example Trading LLC aus Russland. "
    "Transaktionsvolumen 18.000 EUR. "
    "Geschäftszweck: Einkauf von Industriekomponenten. "
    "Zahlungskontext: reguläre Überweisung nach Wareneingang. "
    "Es gibt Hinweise auf negative Medienberichte. "
    "Alle Dokumente liegen vor."
)

response_1 = agent.invoke(
    {"messages": [("human", user_input_1)]},
    config=session,
)
mprint("### 1. Aufruf")
mprint("---")
mprint(response_1["messages"][-1].content)

### 1. Aufruf

---

Bevor ich mit der Compliance-Prüfung fortfahre, benötige ich eine Bestätigung, dass eine formale Sanktionsprüfung für den Lieferanten "Example Trading LLC" durchgeführt wurde und ob dabei ein Treffer vorlag. Bitte bestätigen Sie dies.

⬆️ **Guardrail aktiv:** Alle anderen Felder extrahiert der Agent aus dem Fließtext. Nur `sanctions_clearance_confirmed` fragt er explizit nach — dieses Feld kann nicht inferiert werden und ist im SKILL.md als nicht-ableitbar markiert.

Der User bestätigt jetzt die Sanctions-Prüfung — der Agent hat den vollen Kontext durch das Memory:

In [ ]:
# ── Aufruf 2: User bestätigt Sanctions-Prüfung — gleiche session! ────────────
user_input_2 = "Die Sanctions-Prüfung wurde formal durchgeführt, kein Treffer."

response_2 = agent.invoke(
    {"messages": [("human", user_input_2)]},
    config=session,       # ← selbe session wie Aufruf 1
)

mprint("### 2. Aufruf\n---\n" + response_2["messages"][-1].content)

In [ ]:
show_trace("M31-Agent-Skill-Compliance", show_steps=True)

# 7 | Mixed-Model-Pattern: Gate + Writer
---

Bisher hat ein einzelnes Modell alle Aufgaben übernommen: Fließtext verstehen, Tool aufrufen, Entscheidung treffen und Ergebnis formulieren. Das Mixed-Model-Pattern trennt diese Rollen bewusst:

| Rolle | Modell | Warum |
|---|---|---|
| **Gate** (Tool-Calls, Entscheidung) | `o3` | Striktes Instruction-Following, zuverlässige Tool-Nutzung |
| **Writer** (Report-Erstellung) | `gpt-5.1` | Hohe Textqualität für Audit-Dokumente |

Das Gate ruft `compliance_check` auf und liefert ein strukturiertes Ergebnis. Der Writer erhält dieses Ergebnis und formuliert daraus einen professionellen Compliance-Report.

> **Wann lohnt sich das?** Wenn Gate-Logik und Textqualität unterschiedliche Anforderungen haben — typisch in Compliance-, Legal- oder Security-Prozessen.

In [14]:
#@markdown   <p><font size="4" color='green'>  Gate & Writer</font> </br></p>
diagram= '''
flowchart LR
    USER([User-Input]) --> GATE["🔒 Gate\no3"]
    GATE -->|Tool-Call| TOOL["⚙️ compliance_check\ndeterministisch"]
    TOOL --> GATE
    GATE -->|strukturiertes Ergebnis| WRITER["✍️ Writer\ngpt-5.1"]
    WRITER --> REPORT([Compliance-Report])
'''
mermaid(diagram)

**Setup:** Gate-Agent (o3) und Writer-Chain (gpt-5.1) werden getrennt initialisiert.

---

> **Design-Entscheidung: Wo gehört `WRITER.md` hin?**
>
> `WRITER.md` gehört in `06_skill/compliance/` — auf gleicher Ebene wie `SKILL.md`, **nicht** in `05_prompt/`.
>
> | | `05_prompt/` | `06_skill/compliance/` ✅ |
> |---|---|---|
> | Charakter | Modulübergreifende Prompt-Templates | Skill-spezifische Komponenten |
> | Wiederverwendbar ohne Skill? | Ja | Nein — kennt `risk_level`, `APPROVE/REJECT/ESCALATE` |
> | Loader | `load_prompt()` | `load_prompt(url, mode="S")` |
>
> Beide verwenden `load_prompt()` — aber mit unterschiedlichen Argumenten:
> `05_prompt/`-Dateien haben `## system`/`## human`-Sections → `mode="T"` (ChatPromptTemplate).
> Skill-Dateien in `06_skill/` sind plain Markdown → `mode="S"` (String, Frontmatter wird entfernt).
>
> Der Skill bleibt damit **selbstenthalten**: SKILL.md, WRITER.md, references/ und scripts/ bilden eine transportable Einheit.

In [15]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# ── Gate: o3 — striktes Instruction-Following, zuverlässige Tool-Nutzung ──
gate_llm = init_chat_model(JUDGE)          # kein temperature — o3 unterstützt das nicht
gate_memory = MemorySaver()

gate_agent = create_react_agent(
    model=gate_llm,
    tools=[compliance_check, load_reference],
    prompt=skill_system_prompt,
    checkpointer=gate_memory,
)

# ── Writer: gpt-5.1 — Textqualität für Audit-Dokumente ──
writer_llm = init_chat_model(WORKER)

print("Gate-Agent:  o3  (Tools:", [t.name for t in [compliance_check, load_reference]], ")")
print("Writer-LLM:  gpt-5.1  (keine Tools — nur Textgenerierung)")

Gate-Agent:  o3  (Tools: ['compliance_check', 'load_reference'] )
Writer-LLM:  gpt-5.1  (keine Tools — nur Textgenerierung)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# WRITER.md aus dem Skill laden — Frontmatter wird automatisch entfernt
WRITER_SYSTEM = load_prompt(f"{GITHUB_BASE}/WRITER.md", mode="S")

writer_prompt = ChatPromptTemplate.from_messages([
    ("system", WRITER_SYSTEM),
    ("human", (
        "## Skill-Kontext (SKILL.md)\n\n{skill_context}\n\n"
        "## Gate-Output\n\n{gate_output}\n\n"
        "Erstelle den Compliance-Report im Format des Skill-Kontexts."
    )),
])

writer_chain = writer_prompt | writer_llm
print("Writer-Chain bereit: writer_prompt | gpt-5.1")

**Die SKILL.md hat im Mixed-Model-Pattern zwei verschiedene Charaktere:**

| | Gate (o3) | Writer (gpt-5.1) |
|---|---|---|
| **Rolle der SKILL.md** | Steuernder System-Prompt | Referenzkontext |
| **Wirkung** | Imperativ: *"Always call the tool. Never infer."* | Formatgebend: *"Nutze exakt diese Struktur und Terminologie."* |
| **Was sie erzwingt** | Pflichtschritte, Guardrails, Tool-Nutzung | Ausgabeformat, Feldbenennung, Entscheidungssprache |
| **Übergabe via** | `prompt=skill_system_prompt` | `{skill_context}` im Human-Turn |

Der Skill bleibt damit Ende-zu-Ende konsistent — vom Tool-Call bis zum fertigen Audit-Dokument.

In [17]:
import time

# Neue Session — verhindert Überlappung mit vorherigen Runs
mixed_session = {"configurable": {"thread_id": f"mixed-compliance-{int(time.time())}"}}

user_input_1 = (
    "Wir wollen einen neuen Lieferanten onboarden: "
    "Example Trading LLC aus Russland. "
    "Transaktionsvolumen 18.000 EUR. "
    "Geschäftszweck: Einkauf von Industriekomponenten. "
    "Es gibt Hinweise auf negative Medienberichte. "
    "Alle Dokumente liegen vor."
)

# ── Schritt 1: Gate (o3) — Workflow, Tool-Calls, Entscheidung ────────────────
gate_response_1 = gate_agent.invoke(
    {"messages": [("human", user_input_1)]},
    config=mixed_session,
)
gate_output_1 = gate_response_1["messages"][-1].content

mprint("### Gate-Output (o3)")
mprint("---")
mprint(gate_output_1)

# ── Schritt 2: Writer (gpt-5.1) — Report im SKILL.md-Format ─────────────────
# skill_system_prompt aus Zelle 6 wird mitübergeben → SKILL.md-Charakter bleibt erhalten
writer_response_1 = writer_chain.invoke({
    "skill_context": skill_system_prompt,
    "gate_output": gate_output_1,
})

mprint("### Compliance-Report (gpt-5.1)")
mprint("---")
mprint(writer_response_1.content)

### Gate-Output (o3)

---

Um mit der Compliance-Prüfung fortfahren zu können, benötige ich noch eine explizite Bestätigung zu folgendem Punkt:

• Wurde bereits eine formale Sanktionsprüfung (z. B. gegen EU-, US-, UN-Listen) für „Example Trading LLC“ durchgeführt und gab es dabei einen Treffer?  
  – Bitte antworten:  
    – „ja, kein Treffer“  
    – „ja, Treffer“ (inkl. kurzer Angabe)  
    – „nein, noch nicht geprüft“

Ohne diese Information kann ich den Prozess nicht fortsetzen.

### Compliance-Report (gpt-5.1)

---

### Compliance-Entscheidung

- **Fall:** Unternehmen, Example Trading LLC, Land: nicht angegeben, Transaktionsvolumen: nicht angegeben
- **Durchgeführte Prüfungen:** Sanktionsprüfung – noch ausstehend (fehlende Bestätigung); geografische Prüfung – nicht durchgeführt; Transaktionsgrößenprüfung – nicht durchgeführt (kein Volumen angegeben); Dokumentenprüfung – nicht durchgeführt (keine Unterlagen angegeben)
- **Risikoniveau:** **high** (Systemstatus: Prozess kann wegen fehlender Pflichtangabe nicht fortgesetzt werden)
- **Entscheidung:** **ESCALATE** (bzw. Rückfrage vor Entscheidung)
- **Begründung:**  
  - Kritische Pflichtangabe „sanctions_clearance_confirmed“ fehlt: Es liegt keine explizite Bestätigung vor, ob eine formale Sanktionsprüfung gegen relevante Sanktionslisten (z. B. EU, US, UN) für „Example Trading LLC“ durchgeführt wurde und ob ein Treffer vorliegt.  
  - Ohne diese Information kann das Risiko gemäß Vorgabe nicht bewertet und keine regelkonforme Go/No-Go-Entscheidung getroffen werden.  
  - Weitere Kerndaten (Land, Transaktionsvolumen, Zweck, Zahlungs-/Mittelherkunft) sind ebenfalls nicht spezifiziert und verhindern eine vollständige Prüfung.
- **Fehlende Informationen oder Eskalationspunkt:**  
  - Explizite Aussage zur Sanktionsprüfung für „Example Trading LLC“ (Wortlaut z. B.: „ja, kein Treffer“ / „ja, Treffer (Details …)“ / „nein, noch nicht geprüft“).  
  - Zusätzlich noch fehlend für eine vollständige Bewertung: Land der Geschäftstätigkeit, geplantes Transaktionsvolumen, Geschäfts-/Zahlungszweck, Kontext/Quelle der Mittel (sofern Zahlungen geprüft werden sollen).  
  - Solange insbesondere die Sanktionsbestätigung fehlt, ist der Prozess aus Compliance-Sicht zu pausieren bzw. an eine menschliche Stelle zur Klärung zu eskalieren.
- **Prüfungsnotiz:** Bewertung auf Basis eines simulierten Compliance-Gates ohne Anbindung an Live-Sanktionslisten; Entscheidungsgrundlage sind die im Skill-Kontext definierten Mindestangaben und Prozessregeln. Eine endgültige Risikoeinstufung und Freigabe ist erst nach expliziter Bestätigung der erfolgten Sanktionsprüfung und Vervollständigung der Kerndaten zulässig.

In [ ]:
# ── Aufruf 2: Sanctions-Bestätigung — gleiche Gate-Session ──────────────────
user_input_2 = "Die Sanctions-Prüfung wurde formal durchgeführt, kein Treffer."

gate_response_2 = gate_agent.invoke(
    {"messages": [("human", user_input_2)]},
    config=mixed_session,     # ← selbe session wie Aufruf 1
)
gate_output_2 = gate_response_2["messages"][-1].content

mprint("### Gate-Output (o3) — 2. Aufruf\n---\n" + gate_output_2)

writer_response_2 = writer_chain.invoke({
    "skill_context": skill_system_prompt,
    "gate_output": gate_output_2,
})

mprint("### Compliance-Report (gpt-5.1) — 2. Aufruf\n---\n" + writer_response_2.content)

In [ ]:
show_trace("M31-Agent-Skill-Compliance", show_steps=True)

**Vergleich: Single-Model vs. Mixed-Model**

| Kriterium | Single-Model (`gpt-4o-mini`) | Mixed-Model (`o3` + `gpt-5.1`) |
|---|---|---|
| Tool-Zuverlässigkeit | mittel | hoch (o3) |
| Instruction-Following | mittel | hoch (o3) |
| Report-Textqualität | mittel | hoch (gpt-5.1) |
| Komplexität | gering | mittel |
| Kosten | niedrig | höher |

**Fazit:** Mixed-Model lohnt sich, wenn Gate-Zuverlässigkeit und Dokumentationsqualität beide kritisch sind — typisch in Compliance-, Legal- und Audit-Prozessen. Für Demos und Lernzwecke reicht ein Single-Model.

# 8 | Fazit
---

Dieses Beispiel zeigt die saubere Rollenverteilung eines Skills:
- `SKILL.md` steuert das Verhalten des Agenten
- `references/` halten Fachregeln und Beispiele aus dem Kernkontext heraus
- `scripts/` übernehmen deterministische Teilaufgaben

Genau deshalb passt das Thema in den Kurs **Agenten**: Es geht nicht nur um Antworten, sondern um **zuverlässig gesteuerte agentische Prozesse**.

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



<p><font color='black' size="5">
Skill erweitern
</font></p>

Erweitern Sie den Compliance-Skill um einen zweiten Entscheidungspfad, zum Beispiel für Lieferanten-Onboarding oder Auszahlungsfreigaben.

**Teilaufgaben:**
- Ergänzen Sie mindestens eine neue Regel in `references/risk_rules.md`
- Fügen Sie ein weiteres Beispiel in `references/examples.md` hinzu
- Passen Sie das Skript so an, dass die neue Regel in die Bewertung einfließt
